# FD-31: Финальные прогоны best-конфига на 3 сидах (42/43/44) — Kaggle 2×T4
- **Конфиг:** `config/train_best.yaml` — победитель FD-24 (slot4_full: `microsoft/unixcoder-base`, 1024, `full`, lr 2e-5, 3 эпохи, eff_batch 32). Меняется **только `--seed`**.
- **Инвариант выбора:** строго max PR-AUC на реальных строках `val` (тесты и gold — только отчётность, не отбор). Пороги/калибровка — задача FD-25, здесь не трогаем.
- **Без экспериментов E2/E3:** только финал (3× train → predict → evaluate) + сводная таблица mean±std.
- **Бюджет:** ~27 мин/прогон на 2×T4 → ~1.5 ч суммарно. Диск: 3× ~500 МБ `best/` + артефакты, лимит 20 ГБ не превышается (промежуточные `checkpoint-*` удаляются после каждого сида).
- **Пин кода:** коммит `c512dd7` (freeze-1: финал 3 сидов).

In [ ]:
%cd /kaggle/working
import os
from kaggle_secrets import UserSecretsClient
try:
    _tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    _repo = f"https://{_tok}@github.com/o1uh/FreshDoc.git"
    print("clone: private (token)")
except Exception:
    _repo = "https://github.com/o1uh/FreshDoc.git"
    print("clone: public fallback (repo must be public)")

!if [ ! -d FreshDoc ]; then git clone --quiet {_repo} > /tmp/clone.log 2>&1 || (echo "CLONE FAILED:"; sed -E 's#(https?://)[^@]+@#\1***@#g' /tmp/clone.log); fi
if '_tok' in locals(): del _tok
if '_repo' in locals(): del _repo

%cd /kaggle/working/FreshDoc
!git fetch origin feat/FD-31-freeze-final-runs 2>/dev/null || true
!git checkout c512dd7
!git log --oneline -1

# Рецепт FD-24: базовый GPU-образ Kaggle НЕ трогаем — без `pip install -r requirements.txt`.
# requirements.txt поднимает torch до 2.14.1 и разбивает согласованную связку
# torch/torchvision/torchaudio образа: transformers.audio_utils падает на `import torchaudio`
# (CUDA 13.0 vs 12.8), за ним падает `from transformers import Trainer` в train_m2.py.
# torch/transformers остаются из базового образа — на нём же обучался slot4_full (train_best.yaml),
# совместимость гарантирована. Только guard: дотянуть transformers>=4.48 БЕЗ зависимостей
# (--no-deps), чтобы не задеть torch-стек. Доп. uninstall/пины torchvision/torchaudio
# и TRANSFORMERS_NO_* в FD-24 не использовались — здесь тоже не нужны.
import os
os.environ["NCCL_P2P_DISABLE"] = "1"
!python -c "import transformers, sys; from packaging import version; sys.exit(0 if version.parse(transformers.__version__) >= version.parse('4.48.0') else 1)" || pip install --quiet --no-deps "transformers>=4.48.0"
!python -c "import torch, transformers; from transformers import Trainer; print('env OK:', torch.__version__, transformers.__version__)"
!nvidia-smi

In [ ]:
import os, glob
import pandas as pd
_cands = glob.glob("/kaggle/input/**/dataset_v1.parquet", recursive=True)
assert _cands, "dataset_v1.parquet не найден в /kaggle/input (добавь датасет freshdoc-dataset в Inputs блокнота)"
_p = _cands[0]
os.environ["DATA_PATH"] = _p
print(f"Dataset path: {_p}")

_df = pd.read_parquet(_p, columns=["split"])
assert len(_df) == 27203, f"dataset rows {len(_df)} != 27203"
_exp = {"train": 21997, "val": 559, "test_real": 2895, "test_synth": 1552, "gold": 200}
_got = _df["split"].value_counts().to_dict()
assert _got == _exp, f"split mismatch: {_got}"
print("dataset OK:", _got)
del _df

# Валидация датасета (FD-13): 16 проверок, ok:true. Пулы отсутствуют в Kaggle — проверка пула пропускается автоматически.
!python scripts/validate_dataset.py --in "$DATA_PATH"

In [ ]:
# Генерация бейзлайнов M0 и M1 (вход для evaluate.py; в .gitignore — после clone их нет, регенерируем ~20 c)
!python scripts/baselines_m0_m1.py --dataset "$DATA_PATH" \
  --out_m0 /kaggle/working/m0.csv --out_m1 /kaggle/working/m1.csv \
  --out_model /kaggle/working/m1_hgb.joblib --out_report /kaggle/working/m1_report.json

## Цикл 3 сидов: обучение best-конфига (train → очистка → predict → evaluate)

In [ ]:
# ==============================================================================
# FD-31 финал: 3 прогона config/train_best.yaml, отличается только --seed.
# eff_batch 32 = per_device 8 x accum 2 x 2 GPU (из train_best.yaml, не менять).
# ~27 мин/прогон на 2xT4 (эталон slot4: train_time_sec=1627.5).
# ==============================================================================
import os, subprocess
data_path = os.environ.get("DATA_PATH")
assert data_path and os.path.exists(data_path), "DATA_PATH не задан — выполни ячейку проверки датасета"
for _seed in (42, 43, 44):
    _out = f"/kaggle/working/checkpoints/fd31_seed{_seed}"
    _cmd = (
        f'torchrun --nproc_per_node=2 scripts/train_m2.py --config config/train_best.yaml '
        f'--data_path "{data_path}" --seed {_seed} '
        f'--output_dir "{_out}" --runs_csv reports/runs.csv'
    )
    print(f"=== seed {_seed}: обучение ===\n{_cmd}", flush=True)
    subprocess.run(_cmd, shell=True, check=True)
    subprocess.run(f"rm -rf {_out}/checkpoint-*", shell=True, check=True)
    subprocess.run(f"ls -lh {_out}/best", shell=True, check=True)

In [ ]:
# Инференс + оценка каждого сида (бутстрэп n_boot=1000, как в слотах FD-24)
import os, subprocess
data_path = os.environ.get("DATA_PATH")
for _seed in (42, 43, 44):
    _out = f"/kaggle/working/checkpoints/fd31_seed{_seed}"
    subprocess.run(
        f'python scripts/predict_m2.py --checkpoint "{_out}/best" '
        f'--dataset "{data_path}" --out "{_out}/preds.csv"',
        shell=True, check=True)
    subprocess.run(
        f'python scripts/evaluate.py --dataset "{data_path}" '
        f'--pred M0=/kaggle/working/m0.csv --pred M1=/kaggle/working/m1.csv --pred M2="{_out}/preds.csv" '
        f'--n_boot 1000 --out_metrics "{_out}/metrics.json" --out_table "{_out}/table_e1.csv" '
        f'--figdir "{_out}/figures_e1"',
        shell=True, check=True)
    print(f"=== seed {_seed}: table_e1.csv (первые 25 строк) ===")
    subprocess.run(f"head -n 25 {_out}/table_e1.csv", shell=True, check=True)

In [ ]:
# Сводная таблица mean±std по 3 сидам: PR-AUC (group=all) на val / test_real / gold.
# val/all — метрика выбора (весь val реальный); test_real/gold — только отчётность.
import pandas as pd
_rows = []
for _s in (42, 43, 44):
    _t = pd.read_csv(f"/kaggle/working/checkpoints/fd31_seed{_s}/table_e1.csv")
    _t = _t[(_t["group"] == "all") & (_t["model"] == "M2")]
    for _split in ("val", "test_real", "gold"):
        _r = _t[_t["split"] == _split].iloc[0]
        _rows.append({"seed": _s, "split": _split,
                      "pr_auc": _r["pr_auc"], "pr_auc_lo": _r["pr_auc_lo"],
                      "pr_auc_hi": _r["pr_auc_hi"], "f1": _r["f1"]})
_d = pd.DataFrame(_rows)
print("=== PR-AUC по сидам (group=all) ===")
print(_d.pivot(index="split", columns="seed", values="pr_auc").to_string())
_summary = _d.groupby("split")["pr_auc"].agg(["mean", "std"]).reset_index()
print("\n=== mean±std PR-AUC (3 сида) ===")
print(_summary.to_string(index=False))
_d.to_csv("/kaggle/working/fd31_seeds_summary.csv", index=False)
print("\nsummary: /kaggle/working/fd31_seeds_summary.csv")

In [ ]:
# Упаковка артефактов для скачивания (Output панель Kaggle)
!cd /kaggle/working && tar -czf fd31_final3seeds_artifacts.tar.gz \
  FreshDoc/reports/runs.csv \
  checkpoints/fd31_seed42/best checkpoints/fd31_seed42/preds.csv \
  checkpoints/fd31_seed42/metrics.json checkpoints/fd31_seed42/table_e1.csv \
  checkpoints/fd31_seed43/best checkpoints/fd31_seed43/preds.csv \
  checkpoints/fd31_seed43/metrics.json checkpoints/fd31_seed43/table_e1.csv \
  checkpoints/fd31_seed44/best checkpoints/fd31_seed44/preds.csv \
  checkpoints/fd31_seed44/metrics.json checkpoints/fd31_seed44/table_e1.csv \
  fd31_seeds_summary.csv m0.csv m1.csv
!ls -lh /kaggle/working/fd31_final3seeds_artifacts.tar.gz
!du -sh /kaggle/working/checkpoints